# Do our uncertainty intervals tell the truth?

## Conformal calibration, exchangeability and economic decisions

Previous experiments assessed the predictive variability of estimated conditional benefits. This fifth notebook measures *empirical interval calibration* over independently regenerated datasets, not merely the width of uncertainty bands.

We compare an in-sample Gaussian plug-in prediction interval with an independently calibrated split-conformal interval. Their targets are the noisy future outcome \(Y\), **not** a regression coefficient, conditional mean, or causal treatment effect.

The central questions are:

1. Does observed interval coverage track its nominal level?
2. How much does calibration-set size matter?
3. Does the answer survive MAR predictor missingness?
4. What happens when test observation noise differs from calibration noise?
5. Can interval-based conservative decisions improve economic regret?

The conformal guarantee is *marginal* under exchangeability. It does not guarantee coverage conditional on features, protected groups or economic decisions.

## 1. Calibration requires an untouched partition

Fit the model on \(D_{\mathrm{fit}}\), select among candidates using only that fitting partition, then calibrate on a disjoint \(D_{\mathrm{cal}}\). A third, independent set \(D_{\mathrm{test}}\) is reserved exclusively for evaluation.

For absolute residual scores \(S_i=|Y_i-\widehat m(X_i)|\), the finite-sample split-conformal radius is

\[
q_\alpha=S_{(k)}, \qquad
k=\left\lceil(n_{\mathrm{cal}}+1)(1-\alpha)\right\rceil.
\]

The interval is

\[
C_\alpha(x)=[\widehat m(x)-q_\alpha,\widehat m(x)+q_\alpha].
\]

If \(k>n_{\mathrm{cal}}\), the conservative radius is infinite. The exchangeability argument concerns the held-out scores and yields coverage of *observed outcomes*, with ties making coverage conservative.

This is distinct from the Gaussian plug-in benchmark using training residual standard deviation and a normal critical value. That benchmark ignores estimation effects and need not achieve its nominal level.

In [ ]:
from __future__ import annotations

import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from calibration_analysis import (
    aggregate_coverage,
    run_calibration_experiment,
)

REPEATS: int = int(os.environ.get("CALIBRATION_REPEATS", "12"))
N_FIT: int = int(os.environ.get("CALIBRATION_N_FIT", "180"))
N_CAL: tuple[int, ...] = tuple(
    int(n.strip()) for n in os.environ.get("CALIBRATION_N_CAL", "60,140").split(",")
)
N_TEST: int = int(os.environ.get("CALIBRATION_N_TEST", "250"))
SMOKE: bool = os.environ.get("CALIBRATION_SMOKE", "0") == "1"
ALPHAS = (0.1,) if SMOKE else (0.05, 0.1, 0.2)
SCENARIOS = ("low_variance_signal",) if SMOKE else (
    "low_variance_signal", "high_variance_signal"
)
STRESSES = ("clean",) if SMOKE else ("clean", "mar")
if REPEATS < 1 or N_FIT < 64 or min(N_CAL) < 20 or N_TEST < 20:
    raise ValueError("Invalid simulation settings")

## 2. Independent Monte Carlo coverage

The data-generating processes again include a response driven by a low-variance predictor contrast and one driven by a high-variance direction. We also include MAR missing predictors with an always-observed conditioning variable.

In addition to independent and identically distributed test observations, we evaluate a deliberate **violation of exchangeability**: the observed test noise is multiplied by two while calibration noise stays unchanged. The conditional mean is held fixed.

Model selection happens strictly within fitting observations. Conformal calibration never sees the fitting outcomes used to estimate model parameters, and no model choice uses calibration performance.

In [ ]:
intervals, decisions = run_calibration_experiment(
    repeats=REPEATS,
    n_fit=N_FIT,
    n_cal_values=N_CAL,
    n_test=N_TEST,
    alphas=ALPHAS,
    scenarios=SCENARIOS,
    stresses=STRESSES,
    seed=2026,
)
coverage = aggregate_coverage(intervals, bootstrap_draws=400)
display(coverage.head(12).round(3))
assert intervals["coverage"].between(0, 1).all()
assert np.isfinite(decisions["regret"]).all()

## 3. Does nominal coverage match observed coverage?

Coverage intervals for Monte Carlo means resample *independent generated datasets*. They do not treat rows from one fitted predictor, different model choices or different costs as independent replications.

With fewer than three independently regenerated experiments, the notebook suppresses bootstrap Monte Carlo intervals. Even 12 replicates are exploratory and not enough for precise calibration claims.

In [ ]:
selected = coverage.loc[
    (coverage["model"] == "Fit-only selected")
    & (coverage["scenario"] == "low_variance_signal")
    & (coverage["stress"] == "clean")
    & (coverage["alpha"] == 0.1)
    & (coverage["n_cal"] == N_CAL[0])
]
display(selected[[
    "method", "distribution", "target_coverage", "mean_coverage",
    "coverage_error", "mean_interval_width", "replicates"
]].round(3))
fig, ax = plt.subplots(figsize=(8, 4))
for method, group in selected.groupby("method"):
    ax.plot(
        group["distribution"], group["mean_coverage"],
        marker="o", label=method,
    )
ax.axhline(0.9, linestyle="--", color="grey", label="Nominal 90%")
ax.set(ylabel="Empirical test coverage", xlabel="Test outcome regime",
       title="Independent-test empirical coverage", ylim=(0, 1.02))
ax.legend()
fig.tight_layout()
plt.show()

## 4. Marginal calibration can conceal subgroup differences

Even when marginal coverage is near 90%, intervals may cover poorly for subsets of cases. We report empirical coverage separately for observations with a positive or non-positive value of the fully observed anchor variable \(X_3\).

These are *diagnostics*, not subgroup-conditional guarantees. Stronger subgroup-specific methods require additional calibration design and enough observations in each subgroup.

In [ ]:
display(selected[[
    "method", "distribution",
    "anchor_positive_mean_coverage",
    "anchor_nonpositive_mean_coverage",
]].round(3))

## 5. Economic decisions: coverage is not decision validity

We compare three decision rules at known realised intervention costs:

- **Point threshold:** act if the predicted conditional benefit exceeds the cost.
- **Conservative:** act only if the interval lower bound exceeds the cost.
- **Ideal review:** send cases whose interval straddles the cost to an external reviewer, charging a fixed review cost.

The last rule is explicitly *oracle-aided*: the reviewer is assumed to learn the true conditional expected benefit. It is an optimistic upper-bound benchmark, not an executable policy.

The intervals were calibrated for the noisy future outcome \(Y\), not for its conditional expectation \(m(X)\). Using their lower bounds as intervention criteria does not carry a conformal guarantee of making profitable decisions. Prediction, uncertainty calibration and causal identification remain separate questions.

In [ ]:
decision_summary = (
    decisions.groupby(["model", "stress", "n_cal", "method", "rule", "cost"], as_index=False)
    .agg(
        mean_regret=("regret", "mean"),
        mean_review_rate=("review_rate", "mean"),
        mean_action_rate=("action_rate", "mean"),
    )
)
example = decision_summary.loc[
    (decision_summary["model"] == "Fit-only selected")
    & (decision_summary["stress"] == "clean")
    & (decision_summary["n_cal"] == N_CAL[0])
]
display(example.sort_values(["cost", "mean_regret"]).round(3))
fig, ax = plt.subplots(figsize=(8, 4))
for (method, rule), group in example.groupby(["method", "rule"]):
    if rule == "Ideal review":
        continue
    ax.plot(group["cost"], group["mean_regret"], marker="o",
            label=f"{method}: {rule}")
ax.set(xlabel="Intervention cost", ylabel="Expected regret",
       title="Interval decision rules and economic losses")
ax.legend(fontsize="small")
fig.tight_layout()
plt.show()

## 6. Conclusions and next research steps

The conformal coverage statement applies to an exchangeable future noisy observation and is **marginal over the calibration/test data distribution**. It does not guarantee:

- 90% coverage in every feature subgroup or at each individual case.
- Correct predictions under noise inflation or other deployment distribution shifts.
- A 90% probability that a regression model is correct.
- Profitable actions with a 90% probability.
- Causal treatment-effect identification.

The Gaussian baseline is deliberately naive. More refined parametric prediction intervals, heteroskedasticity-aware conformal procedures, weighted conformal under known covariate shift, and principled abstention with real review costs remain open extensions.

To make defensible empirical claims, increase the independent simulation replicates, present uncertainty around *coverage estimates*, and report failures alongside successful scenarios. For genuine economic or agricultural interventions, identifying the causal response and the actual utility function is prerequisite.